# Llama 3.1 8B Instruct — Dynamic Few-Shot

Runs **dynamic few-shot** and **dynamic few-shot CoT** on the 100-question `facebook/natural_reasoning` test set. Each question's demonstrations are the top-3 semantically most similar pool examples from `dynamic_examples.jsonl`, instead of one fixed hand-written set.

**Model**: `meta-llama/Llama-3.1-8B-Instruct` (4-bit quantized). Results are downloaded locally as `results_llama_dynamic.json`.

In [1]:
!pip install "transformers>=5.10.2" accelerate bitsandbytes
!pip install rouge-score bert-score tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7126d7f7ed28c9f38154ee2bf4d4c8c3fd8085e82a5a01791bbfd13f791d8ea7
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import json
import re
import time
import torch
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
from bert_score import score as compute_bert_score
from huggingface_hub import login
from google.colab import drive, userdata, files

In [3]:
login(token=userdata.get("HF_TOKEN"))
print("Logged into HuggingFace")

Logged into HuggingFace


## Configuration

In [4]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_TAG = "llama"
DRIVE_DATA_PATH = "/content/drive/MyDrive/NLP Project/Data"
SEED = 42
ANSWER_MARKER = "ANSWER:"

TECHNIQUES = ["dynamic_few_shot", "dynamic_few_shot_cot"]

MAX_TOKENS_BY_TYPE = {
    "single_word": 256,
    "short": 512,
    "long": 1024,
    "no_answer": 1024,
}

MAX_INPUT_LENGTH = 3072

In [5]:
def get_batch_size(avg_input_tokens):
    if avg_input_tokens < 500:
        return 12
    elif avg_input_tokens < 1000:
        return 8
    elif avg_input_tokens < 1500:
        return 6
    elif avg_input_tokens < 2000:
        return 4
    elif avg_input_tokens < 2500:
        return 3
    else:
        return 2

## Dataset & Retrieved Examples

In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
with open(f'{DRIVE_DATA_PATH}/sampled.jsonl', 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

ref_70b = {
    item["sample_id"]: item["responses"][0]["response"]
    for item in dataset if item.get("responses")
}

dynamic_examples = {}
with open(f'{DRIVE_DATA_PATH}/dynamic_examples.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        dynamic_examples[rec["sample_id"]] = rec["examples"]

assert len(dataset) == 100, f"Expected 100 test questions, got {len(dataset)}"
missing = [it['sample_id'] for it in dataset if it['sample_id'] not in dynamic_examples]
assert not missing, f"Missing dynamic examples for sample_ids: {missing}"
bad = {sid: len(exs) for sid, exs in dynamic_examples.items() if len(exs) != 3}
assert not bad, f"Expected 3 examples per question, got: {bad}"

sims = [ex['similarity'] for exs in dynamic_examples.values() for ex in exs]
print(f"Loaded {len(dataset)} questions, 3 retrieved examples each")
print(f"70B reference responses available: {len(ref_70b)}")
print(f"Example similarity: mean={sum(sims)/len(sims):.3f}, min={min(sims):.3f}, max={max(sims):.3f}")

Loaded 100 questions, 3 retrieved examples each
70B reference responses available: 100
Example similarity: mean=0.559, min=0.314, max=0.931


In [8]:
item = dataset[0]
print(f"TEST QUESTION (sample_id={item['sample_id']}, type={item['answer_type']}):")
print(item['question'][:400])
print()
for i, ex in enumerate(dynamic_examples[item['sample_id']], 1):
    print(f"--- Retrieved example {i} (pool_id={ex['pool_id']}, similarity={ex['similarity']:.3f}) ---")
    print('Q:', ex['question'][:200])
    print('A:', ex['answer'][:200])
    print()

TEST QUESTION (sample_id=0, type=short):
Given the information about Company B and Company D, including their contribution margins and fixed costs, calculate the breakeven point for each company and determine which company would reach its breakeven point first if both companies started selling their product at the same time.

--- Retrieved example 1 (pool_id=17292, similarity=0.550) ---
Q: A company has annual fixed costs of $100,000, variable costs of $5 per unit, and sells its product for $10 per unit. If the company operates at a break-even point, what is the volume of sales in units
A: 20,000 units, $200,000

--- Retrieved example 2 (pool_id=16851, similarity=0.699) ---
Q: Compute the company's break-even point in units, given the sales, variable costs, and fixed costs. Show your work and explain your reasoning.
A: 40,000

--- Retrieved example 3 (pool_id=3519, similarity=0.720) ---
Q: Suppose a company produces a product with a sale price of $30 per unit and a variable cost of $20

## Model

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded: meta-llama/Llama-3.1-8B-Instruct
GPU memory used: 6.06 GB
GPU memory reserved: 6.20 GB


## Prompting Techniques (dynamic examples)

In [10]:
def build_messages(question, technique, sample_id):
    examples = dynamic_examples[sample_id]

    if technique == "dynamic_few_shot":
        messages = [
            {"role": "system", "content": "You are a knowledgeable assistant. Answer questions thoroughly. State your final answer on a new line starting with 'ANSWER:'"}
        ]
        for ex in examples:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"ANSWER: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

    elif technique == "dynamic_few_shot_cot":
        messages = [
            {"role": "system", "content": "You are a knowledgeable assistant. Think through problems step by step. State your final answer on a new line starting with 'ANSWER:'"}
        ]
        for ex in examples:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"{ex['reasoning']}\n\nANSWER: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

In [11]:
print(f"Prompt token lengths (limit {MAX_INPUT_LENGTH}):")
for technique in TECHNIQUES:
    lengths = []
    for item in dataset:
        msgs = build_messages(item['question'], technique, item['sample_id'])
        templated = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        lengths.append(len(tokenizer.encode(templated)))
    over = sum(1 for l in lengths if l > MAX_INPUT_LENGTH)
    print(f"  {technique}: mean={sum(lengths)/len(lengths):.0f}, max={max(lengths)}, over limit={over}")
    assert over == 0, f"{over} prompts exceed MAX_INPUT_LENGTH and would be truncated!"
print("All prompts fit — nothing will be truncated.")

Prompt token lengths (limit 3072):
  dynamic_few_shot: mean=481, max=796, over limit=0
  dynamic_few_shot_cot: mean=1094, max=1565, over limit=0
All prompts fit — nothing will be truncated.


## Inference

In [12]:
def generate_batch(messages_list, max_new_tokens=1024):
    templated = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]
    inputs = tokenizer(
        templated, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_INPUT_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

    responses = []
    for i, output in enumerate(outputs):
        prompt_len = inputs["input_ids"][i].shape[0]
        decoded = tokenizer.decode(output[prompt_len:], skip_special_tokens=True)
        responses.append(decoded.strip())
    return responses

In [13]:
results = []

for technique in TECHNIQUES:
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"Running: {technique}")
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"{'='*60}")

    all_messages = [
        build_messages(item["question"], technique, item["sample_id"])
        for item in dataset
    ]
    templated_for_len = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in all_messages
    ]
    input_lengths = [len(tokenizer.encode(t)) for t in templated_for_len]

    sorted_indices = sorted(range(len(dataset)), key=lambda i: input_lengths[i])
    all_messages = [all_messages[i] for i in sorted_indices]
    sorted_items = [dataset[i] for i in sorted_indices]
    sorted_lengths = [input_lengths[i] for i in sorted_indices]

    avg_len = sum(sorted_lengths) / len(sorted_lengths)
    batch_size = get_batch_size(avg_len)
    print(f"Avg input tokens: {avg_len:.0f} \u2192 batch size: {batch_size}")

    for batch_start in tqdm(range(0, len(dataset), batch_size), desc=technique):
        batch_end = min(batch_start + batch_size, len(dataset))
        batch_messages = all_messages[batch_start:batch_end]
        batch_items = sorted_items[batch_start:batch_end]

        batch_max_tokens = max(
            MAX_TOKENS_BY_TYPE.get(item["answer_type"], 1024)
            for item in batch_items
        )

        start = time.time()
        responses = generate_batch(batch_messages, max_new_tokens=batch_max_tokens)
        gen_time = (time.time() - start) / len(responses)

        for item, response in zip(batch_items, responses):
            results.append({
                "sample_id": item["sample_id"],
                "question": item["question"],
                "reference_answer": item["reference_answer"],
                "answer_type": item["answer_type"],
                "technique": technique,
                "response": response,
                "generation_time": gen_time,
                "response_length": len(response.split()),
            })

df = pd.DataFrame(results)
print(f"\nDone: {len(df)} results ({len(dataset)} questions x {len(TECHNIQUES)} techniques)")


Running: dynamic_few_shot
GPU memory: 6.06 GB
Avg input tokens: 481 → batch size: 12


dynamic_few_shot:   0%|          | 0/9 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Running: dynamic_few_shot_cot
GPU memory: 6.07 GB
Avg input tokens: 1094 → batch size: 6


dynamic_few_shot_cot:   0%|          | 0/17 [00:00<?, ?it/s]


Done: 200 results (100 questions x 2 techniques)


In [14]:
out_path = f"/content/results_{MODEL_TAG}_dynamic.json"
df.to_json(out_path, orient="records", indent=2)
files.download(out_path)
print(f"Saved and downloading: {out_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloading: /content/results_llama_dynamic.json


In [15]:
del model
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU memory after cleanup: 3.97 GB


## Answer Extraction & Normalization

In [16]:
def normalize_latex(text):
    text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', text)
    text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', text)
    text = re.sub(r'\$+', '', text)
    text = re.sub(r'\\(left|right|quad|qquad|text|mathrm|mathbf|displaystyle)\b', '', text)
    text = re.sub(r'\\{2,}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_response(response):
    response = re.sub(r'^#+\s*(Step\s*\d+|Solution|Answer|Explanation)[:\s]*', '', response, flags=re.MULTILINE)
    response = re.sub(r"^(I'd be happy to help!?|Sure!?|Here's? (?:the|my) (?:answer|solution|response)[:\s]*)", '', response, flags=re.IGNORECASE)
    return response.strip()

def extract_answer(response):
    if ANSWER_MARKER in response:
        return response.split(ANSWER_MARKER)[-1].strip()

    boxed = re.findall(r'\\boxed\{([^}]+)\}', response)
    if boxed:
        return boxed[-1]

    patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]+(.+?)(?:\.|$)',
        r'therefore[,:\s]+(.+?)(?:\.|$)',
    ]
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()

    sentences = [s.strip() for s in response.rstrip('.').split('.') if s.strip()]
    return sentences[-1] if sentences else response

def normalize(text):
    text = normalize_latex(text)
    return text.lower().strip().rstrip('.')

## Evaluation

In [17]:
rouge_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def exact_match(prediction, reference):
    return normalize(extract_answer(prediction)) == normalize(reference)

def contains_match(prediction, reference):
    return normalize(reference) in normalize(prediction)

def compute_rouge_l(prediction, reference):
    return rouge_l_scorer.score(reference, prediction)['rougeL'].fmeasure

def compute_f1_token(prediction, reference):
    pred_tokens = normalize(extract_answer(prediction)).split()
    ref_tokens = normalize(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    common = sum((pred_counts & ref_counts).values())
    if not common:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

In [18]:
has_ref = df[df['answer_type'] != 'no_answer'].copy()

has_ref['extracted_answer'] = has_ref['response'].apply(extract_answer)

has_ref['exact_match'] = has_ref.apply(
    lambda r: exact_match(r['response'], r['reference_answer']), axis=1
)
has_ref['contains_match'] = has_ref.apply(
    lambda r: contains_match(r['response'], r['reference_answer']), axis=1
)
has_ref['rouge_l'] = has_ref.apply(
    lambda r: compute_rouge_l(r['response'], r['reference_answer']), axis=1
)
has_ref['rouge_l_extracted'] = has_ref.apply(
    lambda r: compute_rouge_l(r['extracted_answer'], r['reference_answer']), axis=1
)
has_ref['f1_token'] = has_ref.apply(
    lambda r: compute_f1_token(r['response'], r['reference_answer']), axis=1
)

has_ref['has_marker'] = has_ref['response'].str.contains(ANSWER_MARKER, regex=False)
print("ANSWER: marker hit rate by technique:")
print(has_ref.groupby('technique')['has_marker'].mean().round(4))
print()

ANSWER: marker hit rate by technique:
technique
dynamic_few_shot        0.2561
dynamic_few_shot_cot    0.7195
Name: has_marker, dtype: float64



### BERTScore & 70B Reference Comparison

In [19]:
has_ref['ref_70b'] = has_ref['sample_id'].map(ref_70b)
mask_70b = has_ref['ref_70b'].notna()

preds_ref = has_ref['response'].tolist()
refs_ref = has_ref['reference_answer'].tolist()

preds_70b = has_ref[mask_70b]['response'].tolist()
refs_70b_list = has_ref[mask_70b]['ref_70b'].tolist()

all_preds = preds_ref + preds_70b
all_refs = refs_ref + refs_70b_list

print(f"Computing BERTScore for {len(all_preds)} pairs...")
P, R, F1 = compute_bert_score(all_preds, all_refs, lang='en', verbose=True)

n_ref = len(preds_ref)
has_ref['bertscore_f1'] = F1[:n_ref].numpy()
has_ref.loc[mask_70b, 'bertscore_vs_70b'] = F1[n_ref:].numpy()

has_ref.loc[mask_70b, 'rouge_l_vs_70b'] = has_ref[mask_70b].apply(
    lambda r: compute_rouge_l(r['response'], r['ref_70b']), axis=1
)

print("Done!")

Computing BERTScore for 328 pairs...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/6 [00:00<?, ?it/s]

done in 20.90 seconds, 15.69 sentences/sec
Done!


In [20]:
metrics = has_ref.groupby('technique').agg(
    exact_match=('exact_match', 'mean'),
    contains_match=('contains_match', 'mean'),
    rouge_l=('rouge_l', 'mean'),
    bertscore_f1=('bertscore_f1', 'mean'),
    f1_token=('f1_token', 'mean'),
    rouge_l_vs_70b=('rouge_l_vs_70b', 'mean'),
    bertscore_vs_70b=('bertscore_vs_70b', 'mean'),
    avg_response_len=('response_length', 'mean'),
    avg_gen_time=('generation_time', 'mean'),
).round(4)

metrics

,exact_match,contains_match,rouge_l,bertscore_f1,f1_token,rouge_l_vs_70b,bertscore_vs_70b,avg_response_len,avg_gen_time
technique,,,,,,,,,
dynamic_few_shot,0.0,0.0488,0.0852,0.8127,0.1061,0.2568,0.8546,266.5976,45.1976
dynamic_few_shot_cot,0.0,0.0366,0.0734,0.8118,0.1716,0.2922,0.8670,316.8537,79.6239
